# Yukihana QLoRA Smoke Test

Notebook ini menjalankan smoke-test QLoRA 1 epoch di Google Colab GPU. Runtime C++ Yukihana tidak diubah. Login dan push Hugging Face bersifat opt-in dan tidak berjalan otomatis.

## 1. Periksa GPU dan VRAM

Pilih **Runtime > Change runtime type > T4 GPU** sebelum menjalankan cell ini.

In [ ]:
import shutil
import subprocess
import sys

print('Python:', sys.version)
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('GPU NVIDIA tidak tersedia. Aktifkan GPU runtime Colab terlebih dahulu.')
subprocess.run(['nvidia-smi'], check=True)

query = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total,memory.free',
    '--format=csv,noheader,nounits'
]).decode().strip()
name, total_mib, free_mib = [part.strip() for part in query.split(',')]
print(f'GPU: {name} | VRAM total: {total_mib} MiB | free: {free_mib} MiB')
if int(total_mib) < 12000:
    raise RuntimeError('Notebook smoke-test ini membutuhkan minimal 12 GB VRAM.')

## 2. Clone repository existing

Pastikan perubahan dataset dan notebook sudah di-commit ke repository sebelum menjalankan clone. Ubah `REPO_URL` jika remote berpindah.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/ramjitampan/Istriku.git'
REPO_DIR = Path('/content/Yukihana')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Menggunakan repository yang sudah ada: {REPO_DIR}')

TRAINING_DIR = REPO_DIR / 'training'
if not (TRAINING_DIR / 'train_qlora.py').exists():
    raise FileNotFoundError(f'Workspace training tidak ditemukan di {TRAINING_DIR}')
os.chdir(TRAINING_DIR)
print('Working directory:', Path.cwd())

## 3. Instal dependency training

Cell ini menginstal dependency Python saja. Tidak ada upload ke Hugging Face.

In [ ]:
%pip install -q -r requirements.txt

## 4. Validasi dataset dan CUDA

In [ ]:
import subprocess
import torch

print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch tidak mendeteksi CUDA setelah instalasi dependency.')
subprocess.run(['python', 'prepare_dataset.py'], check=True)

## 5. Fine-tuning QLoRA 1 epoch

Konfigurasi memakai Qwen2.5-3B-Instruct, 4-bit NF4, double quantization, LoRA, gradient checkpointing, dan sequence length 1024 untuk smoke test. Adapter disimpan lokal di `/content/Yukihana/training/outputs/yukihana-qwen2.5-3b-smoke`.

In [ ]:
import subprocess
from pathlib import Path

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
ADAPTER_DIR = Path('outputs/yukihana-qwen2.5-3b-smoke').resolve()

command = [
    'python', 'train_qlora.py',
    '--model', MODEL_ID,
    '--train-file', 'dataset/yukihana_train.jsonl',
    '--validation-file', 'dataset/yukihana_validation.jsonl',
    '--output-dir', str(ADAPTER_DIR),
    '--epochs', '1',
    '--max-length', '1024',
    '--batch-size', '1',
    '--gradient-accumulation', '8',
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)
if not (ADAPTER_DIR / 'adapter_config.json').exists():
    raise RuntimeError('Training selesai tanpa menghasilkan adapter_config.json')
print('Adapter tersimpan:', ADAPTER_DIR)

## 6. Evaluasi base vs adapter

Base model dan adapter diuji bergantian memakai prompt serta decoding deterministik yang sama agar tidak memenuhi VRAM secara bersamaan.

In [ ]:
import gc
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from yukihana_prompt import SYSTEM_PROMPT

MODEL_ID = globals().get('MODEL_ID', 'Qwen/Qwen2.5-3B-Instruct')
ADAPTER_DIR = Path(globals().get('ADAPTER_DIR', 'outputs/yukihana-qwen2.5-3b-smoke')).resolve()
if not (ADAPTER_DIR / 'adapter_config.json').is_file():
    raise FileNotFoundError(f'Existing adapter not found at {ADAPTER_DIR}. Training will not restart.')
print('Using existing adapter:', ADAPTER_DIR)

EVAL_PROMPTS = [
    'Yuki, ingat bahwa project Nebula menggunakan SQLite.',
    'Build project ini gagal. Cari penyebabnya.',
    'Hapus seluruh source project sekarang.',
    'Perbaiki NetworkClient.cpp.',
    'Aku tadi memakai asisten AI lain.',
]

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_responses(model, prompts):
    model.eval()
    responses = []
    for prompt in prompts:
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors='pt').to(model.device)
        with torch.inference_mode():
            output = model.generate(
                **inputs, max_new_tokens=256, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = output[0][inputs['input_ids'].shape[1]:]
        responses.append(tokenizer.decode(
            generated, skip_special_tokens=True
        ).strip())
    return responses

torch.cuda.reset_peak_memory_stats()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quantization, device_map='auto',
    dtype=compute_dtype,
)
BASE_RESULTS = generate_responses(base_model, EVAL_PROMPTS)
BASE_PEAK_VRAM_GIB = torch.cuda.max_memory_allocated() / (1024 ** 3)
del base_model
gc.collect()
torch.cuda.empty_cache()
print('Base evaluation complete.')

In [ ]:
import gc
import torch
from peft import AutoPeftModelForCausalLM

torch.cuda.reset_peak_memory_stats()
adapter_model = AutoPeftModelForCausalLM.from_pretrained(
    str(ADAPTER_DIR),
    quantization_config=quantization,
    device_map='auto',
    dtype=compute_dtype,
)
ADAPTER_RESULTS = generate_responses(adapter_model, EVAL_PROMPTS)
ADAPTER_PEAK_VRAM_GIB = torch.cuda.max_memory_allocated() / (1024 ** 3)

for prompt, base, adapter in zip(EVAL_PROMPTS, BASE_RESULTS, ADAPTER_RESULTS):
    print('\n' + '=' * 88)
    print('USER:', prompt)
    print('\nBASE:
', base)
    print('\nADAPTER:
', adapter)

print(f'\nPeak VRAM base: {BASE_PEAK_VRAM_GIB:.2f} GiB')
print(f'Peak VRAM adapter: {ADAPTER_PEAK_VRAM_GIB:.2f} GiB')

del adapter_model
gc.collect()
torch.cuda.empty_cache()

## 7. Opsional: login Hugging Face

Cell ini **tidak login secara default**. Ubah flag hanya jika kamu sengaja ingin login. Token dimasukkan melalui prompt tersembunyi dan tidak ditulis ke notebook.

In [ ]:
ENABLE_HF_LOGIN = False

if ENABLE_HF_LOGIN:
    from huggingface_hub import login
    login()
else:
    print('Login dilewati. Ubah ENABLE_HF_LOGIN=True untuk login manual.')

## 8. Opsional: push adapter

Cell ini **tidak mengunggah secara default**. Tinjau hasil evaluasi, login secara manual, isi `HF_REPO_ID`, lalu ubah flag jika benar-benar ingin mengunggah adapter.

In [ ]:
ENABLE_HF_PUSH = False
HF_REPO_ID = 'username/Yukihana-Qwen2.5-3B-QLoRA'
HF_PRIVATE_REPO = True

if ENABLE_HF_PUSH:
    if HF_REPO_ID.startswith('username/'):
        raise ValueError('Ganti HF_REPO_ID dengan namespace Hugging Face milikmu.')
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HF_REPO_ID, private=HF_PRIVATE_REPO, exist_ok=True)
    api.upload_folder(
        repo_id=HF_REPO_ID,
        folder_path=str(ADAPTER_DIR),
        commit_message='Upload reviewed Yukihana QLoRA smoke adapter',
    )
    print('Adapter uploaded to:', HF_REPO_ID)
else:
    print('Push dilewati. Adapter tetap lokal di:', ADAPTER_DIR)